In [1]:
# ---- CELL 1: Mount Drive, copy dataset to local disk ----
from google.colab import drive
drive.mount('/content/drive')

import shutil, os, time

LOCAL_ROOT = "/content/local_dataset"
DATASET_ROOT_DRIVE = "/content/drive/MyDrive/container-ocr/data/container.v2i.yolov11"
SPLITS_TO_LABEL = ["train", "valid", "test"]

if not os.path.exists(LOCAL_ROOT):
    print("Copying dataset from Drive to local disk...")
    start = time.time()
    for split in SPLITS_TO_LABEL:
        for subfolder in ["images", "labels"]:
            src = os.path.join(DATASET_ROOT_DRIVE, split, subfolder)
            dst = os.path.join(LOCAL_ROOT, split, subfolder)
            shutil.copytree(src, dst)
    print(f"Done in {time.time()-start:.1f}s")
else:
    print("Local copy already exists.")

DATASET_ROOT = LOCAL_ROOT

Mounted at /content/drive
Copying dataset from Drive to local disk...
Done in 349.3s


In [2]:
# ---- CELL 2: Config ----
import csv
import random
from datetime import datetime

OUTPUT_CSV = "/content/drive/MyDrive/container-ocr/training_labels/manual_labels.csv"
BATCH_SIZE = 60          # crops offered per session; re-run Cell 4 for more
RANDOM_SEED = 42

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

In [3]:
# ---- CELL 3: ISO 6346 check-digit validator ----
_LETTER_VALUES = {c: v for c, v in zip(
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ",
    [10,12,13,14,15,16,17,18,19,20,21,23,24,25,26,27,28,29,30,31,32,34,35,36,37,38]
)}

def iso6346_check_digit(code10: str) -> int:
    """code10 = first 10 characters (4 letters + 6 digits).
    Returns expected check digit (0-9)."""
    values = [_LETTER_VALUES[c] for c in code10[:4]] + [int(c) for c in code10[4:10]]
    total = sum(v * (2 ** i) for i, v in enumerate(values))
    remainder = total % 11
    return remainder % 10  # remainder==10 maps to check digit 0

def validate_full_code(code: str):
    """code = full 11 chars (4 letters + 7 digits, last digit is the check digit)."""
    code = code.strip().upper().replace(" ", "")
    if len(code) != 11:
        return False, f"expected 11 characters, got {len(code)}"
    if not code[:4].isalpha():
        return False, "first 4 characters should be letters"
    if not code[4:].isdigit():
        return False, "characters 5-11 should be digits"
    expected = iso6346_check_digit(code[:10])
    actual = int(code[10])
    if expected != actual:
        return False, f"checksum mismatch: expected check digit {expected}, got {actual}"
    return True, "valid"

In [25]:
# ---- CELL 4: Gather candidate boxes from YOLO labels, sample a batch ----

def load_boxes_for_split(split):
    """Returns list of (image_path, box_index, x_c, y_c, w, h, split) for every box."""
    img_dir = os.path.join(DATASET_ROOT, split, "images")
    lbl_dir = os.path.join(DATASET_ROOT, split, "labels")
    candidates = []
    for fname in sorted(os.listdir(img_dir)):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        stem = os.path.splitext(fname)[0]
        lbl_path = os.path.join(lbl_dir, stem + ".txt")
        if not os.path.exists(lbl_path):
            continue
        img_path = os.path.join(img_dir, fname)
        with open(lbl_path) as f:
            for i, line in enumerate(f):
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                _, x_c, y_c, w, h = parts
                candidates.append((img_path, i, float(x_c), float(y_c), float(w), float(h), split))
    return candidates

def load_existing_labels():
    """Returns a set of (image_path, box_index) already labeled, so we skip them."""
    done = set()
    if os.path.exists(OUTPUT_CSV):
        with open(OUTPUT_CSV, newline='') as f:
            for row in csv.DictReader(f):
                done.add((row["image_path"], int(row["box_index"])))
    return done

all_candidates = []
for split in SPLITS_TO_LABEL:
    all_candidates.extend(load_boxes_for_split(split))

already_done = load_existing_labels()
remaining = [c for c in all_candidates if (c[0], c[1]) not in already_done]

random.shuffle(remaining)  # no fixed seed — every re-run gets a genuinely fresh shuffle
batch = remaining[:BATCH_SIZE]

print(f"Total boxes found: {len(all_candidates)}")
print(f"Already labeled:   {len(already_done)}")
print(f"Remaining:         {len(remaining)}")
print(f"This session will offer up to {len(batch)} new crops.")

Total boxes found: 5730
Already labeled:   2005
Remaining:         3725
This session will offer up to 60 new crops.


In [ ]:
# ---- CELL 4b: Safe-video steering (prioritize non-test-leaking source videos) ----
#
# Reuses the exact same video_id derivation + test-split matching logic as
# scripts/paddle_finetune/extract_crops.py's leakage exclusion, via the
# shared scripts/paddle_finetune/video_safety.py module -- so "safe" here
# means exactly the same thing it means at extraction time. Assumes this
# repo (including scripts/paddle_finetune/video_safety.py) is synced to
# Drive under container-ocr/, same as DATASET_ROOT_DRIVE and OUTPUT_CSV
# above.
#
# Does NOT touch Cell 4's dedup logic (`already_done` / `remaining`) above --
# it only reorders the already-deduped `remaining` list so safe-video boxes
# are offered first, before spending labeling time on boxes that can never
# be used for fine-tuning under the current leakage policy. The shuffle
# from Cell 4 is preserved within each safe/unsafe group (stable sort).

import sys
REPO_ON_DRIVE = "/content/drive/MyDrive/container-ocr"
if REPO_ON_DRIVE not in sys.path:
    sys.path.insert(0, REPO_ON_DRIVE)

from scripts.paddle_finetune.video_safety import is_video_safe, build_test_video_ids

TEST_IMAGES_DIR_DRIVE = os.path.join(REPO_ON_DRIVE, "container.v2i.yolov11.resplit", "test", "images")
test_video_ids = build_test_video_ids(TEST_IMAGES_DIR_DRIVE)

remaining.sort(key=lambda c: not is_video_safe(c[0], test_video_ids))  # safe first
batch = remaining[:BATCH_SIZE]

safe_in_remaining = sum(1 for c in remaining if is_video_safe(c[0], test_video_ids))
safe_in_batch = sum(1 for c in batch if is_video_safe(c[0], test_video_ids))
print(f"Safe-video boxes in remaining pool: {safe_in_remaining} / {len(remaining)}")
print(f"This session's batch: {safe_in_batch} safe / {len(batch) - safe_in_batch} unsafe (safe offered first)")


In [23]:
# ---- CELL 5: Interactive labeling loop — ipywidgets version ----
#
# Why this replaced the matplotlib + input() version: input() inside a loop
# is unreliable in Colab (the text box renders then vanishes — an autoscroll
# interaction, confirmed via isolated test). This version creates the
# Text/Button widgets ONCE, and only the image Output area refreshes per
# crop — no blocking input() calls anywhere.
#
# Controls: type the 11-char code and click Submit (or press Enter in the
# text box), or click Skip / Not a code / Clipped / Stop.
# "Clipped" is for codes genuinely cut off by the box edge — never type a
# full confident code if any character was actually inferred/guessed rather
# than clearly read; use Clipped with a partial/? placeholder instead.

import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image

state = {
    "index": 0,
    "labeled_this_session": 0,
    "csv_file": open(OUTPUT_CSV, "a", newline=''),
}
state["writer"] = csv.writer(state["csv_file"])
if os.path.getsize(OUTPUT_CSV) == 0:
    state["writer"].writerow(["image_path", "box_index", "x_center", "y_center", "width",
                               "height", "label_text", "checksum_valid", "split", "status", "timestamp"])
    state["csv_file"].flush()

image_out = widgets.Output()
status_out = widgets.Output()
text_box = widgets.Text(placeholder="Type the container code, e.g. CSQU3054383", layout=widgets.Layout(width="300px"))
btn_submit = widgets.Button(description="Submit", button_style="success")
btn_skip = widgets.Button(description="Skip")
btn_notacode = widgets.Button(description="Not a code")
btn_clipped = widgets.Button(description="Clipped")
btn_stop = widgets.Button(description="Stop", button_style="danger")

controls = widgets.HBox([text_box, btn_submit, btn_skip, btn_notacode, btn_clipped, btn_stop])
display(image_out, controls, status_out)


def show_current_crop():
    with image_out:
        clear_output(wait=True)
        if state["index"] >= len(batch):
            print(f"Batch finished. Labeled {state['labeled_this_session']} crops this session.")
            return
        img_path, box_idx, x_c, y_c, w, h, split = batch[state["index"]]
        img = Image.open(img_path).convert("RGB")
        W, H = img.size
        left = (x_c - w / 2) * W
        top = (y_c - h / 2) * H
        right = (x_c + w / 2) * W
        bottom = (y_c + h / 2) * H
        margin = 0.05
        left = max(0, left - margin * (right - left))
        right = min(W, right + margin * (right - left))
        top = max(0, top - margin * (bottom - top))
        bottom = min(H, bottom + margin * (bottom - top))
        crop = img.crop((left, top, right, bottom))
        display(crop)
        print(f"{os.path.basename(img_path)}  (box {box_idx})  [{split}]"
              f"  —  {state['index'] + 1}/{len(batch)}")
    text_box.value = ""


def write_row(status, label_text="", checksum_valid=""):
    img_path, box_idx, x_c, y_c, w, h, split = batch[state["index"]]
    state["writer"].writerow([img_path, box_idx, x_c, y_c, w, h, label_text,
                               checksum_valid, split, status, datetime.now().isoformat()])
    state["csv_file"].flush()


def advance():
    state["index"] += 1
    show_current_crop()


def on_submit(_):
    entry = text_box.value.strip()
    if not entry:
        return
    is_valid, msg = validate_full_code(entry)
    with status_out:
        clear_output(wait=True)
        if not is_valid:
            print(f"⚠️  {msg} — saved anyway, but double-check this one (checksum_valid=False).")
        else:
            print(f"✓ Saved (checksum_valid=True)")
    write_row("labeled", entry.upper(), is_valid)
    state["labeled_this_session"] += 1
    advance()


def on_skip(_):
    write_row("skip")
    advance()


def on_notacode(_):
    write_row("notacode")
    advance()


def on_clipped(_):
    entry = text_box.value.strip()
    write_row("clipped", entry.upper() if entry else "")
    advance()


def on_stop(_):
    with status_out:
        clear_output(wait=True)
        print(f"Stopped. Labeled {state['labeled_this_session']} crops this session.")
    state["csv_file"].close()


btn_submit.on_click(on_submit)
btn_skip.on_click(on_skip)
btn_notacode.on_click(on_notacode)
btn_clipped.on_click(on_clipped)
btn_stop.on_click(on_stop)
text_box.on_submit(on_submit)  # pressing Enter also submits

show_current_crop()

Output()

Output()

In [24]:
# ---- CELL 6: Sanity check after a session ----
# Run this any time to confirm what's actually saved (matches the pattern
# used on 06/07/26 to verify the in-memory session counter vs. real CSV rows).
import pandas as pd
df = pd.read_csv(OUTPUT_CSV)
print(f"Total rows in CSV: {len(df)}")
print(df['status'].value_counts())

Total rows in CSV: 2012
status
labeled           1979
skip                18
clipped              9
confirmed_asis       6
Name: count, dtype: int64


LABELLING CORRECTION


In [26]:
# ---- CELL 1: Mount Drive, copy dataset to local disk ----
from google.colab import drive
drive.mount('/content/drive')

import shutil, os, time

LOCAL_ROOT = "/content/local_dataset"
DATASET_ROOT_DRIVE = "/content/drive/MyDrive/container-ocr/data/container.v2i.yolov11"
SPLITS_TO_LABEL = ["train", "valid"]

if not os.path.exists(LOCAL_ROOT):
    print("Copying dataset from Drive to local disk...")
    start = time.time()
    for split in SPLITS_TO_LABEL:
        for subfolder in ["images", "labels"]:
            src = os.path.join(DATASET_ROOT_DRIVE, split, subfolder)
            dst = os.path.join(LOCAL_ROOT, split, subfolder)
            shutil.copytree(src, dst)
    print(f"Done in {time.time()-start:.1f}s")
else:
    print("Local copy already exists.")

DATASET_ROOT = LOCAL_ROOT

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Local copy already exists.


In [27]:
# ---- CELL 2: Config ----
import csv
from datetime import datetime

OUTPUT_CSV = "/content/drive/MyDrive/container-ocr/training_labels/manual_labels.csv"
CORRECTION_BATCH_SIZE = 200   # rows offered per session; re-run Cell 4 for more

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

In [28]:
# ---- CELL 3: ISO 6346 check-digit validator ----
_LETTER_VALUES = {c: v for c, v in zip(
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ",
    [10,12,13,14,15,16,17,18,19,20,21,23,24,25,26,27,28,29,30,31,32,34,35,36,37,38]
)}

def iso6346_check_digit(code10: str) -> int:
    """code10 = first 10 characters (4 letters + 6 digits).
    Returns expected check digit (0-9)."""
    values = [_LETTER_VALUES[c] for c in code10[:4]] + [int(c) for c in code10[4:10]]
    total = sum(v * (2 ** i) for i, v in enumerate(values))
    remainder = total % 11
    return remainder % 10  # remainder==10 maps to check digit 0

def validate_full_code(code: str):
    """code = full 11 chars (4 letters + 7 digits, last digit is the check digit)."""
    code = code.strip().upper().replace(" ", "")
    if len(code) != 11:
        return False, f"expected 11 characters, got {len(code)}"
    if not code[:4].isalpha():
        return False, "first 4 characters should be letters"
    if not code[4:].isdigit():
        return False, "characters 5-11 should be digits"
    expected = iso6346_check_digit(code[:10])
    actual = int(code[10])
    if expected != actual:
        return False, f"checksum mismatch: expected check digit {expected}, got {actual}"
    return True, "valid"

In [31]:
# ---- CELL 4: Gather checksum_valid=False rows for correction ----

def load_all_rows():
    """Returns (fieldnames, rows) for the full manual_labels.csv, preserving
    row order and the existing schema untouched (no new columns)."""
    with open(OUTPUT_CSV, newline='') as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)
    return fieldnames, rows

FIELDNAMES, all_rows = load_all_rows()

# Correction candidates: status == "labeled" AND checksum_valid == "False".
# Rows resolved via "Confirm as-is" get status == "confirmed_asis" and are
# excluded here, so they never resurface in a later correction pass.
correction_indices = [
    i for i, row in enumerate(all_rows)
    if row["status"] == "labeled" and row["checksum_valid"] == "False"
]

batch_indices = correction_indices[:CORRECTION_BATCH_SIZE]

print(f"Total rows in CSV:                 {len(all_rows)}")
print(f"checksum_valid=False, unresolved:  {len(correction_indices)}")
print(f"This session will offer up to {len(batch_indices)} for review.")

Total rows in CSV:                 2012
checksum_valid=False, unresolved:  0
This session will offer up to 0 for review.


In [30]:
# ---- CELL 5: Interactive correction loop — ipywidgets version ----
#
# Same mechanism as Labelling.ipynb Cell 5 (persistent widgets, no blocking
# input() calls) — adapted for correcting existing rows instead of labeling
# new ones. Shows the actual crop again (not just the old text), since some
# checksum_valid=False rows are genuine misreads, not typos.
#
# Controls:
#   - Text box is pre-filled with the old label_text — edit it if it's a
#     typo, or clear and retype if you actually misread it, then click
#     Submit (or press Enter).
#   - "Confirm as-is" — for the rare genuinely malformed real-world code;
#     keeps checksum_valid=False permanently but marks the row resolved so
#     it won't be offered again in a future correction pass.
#   - "Skip for now" — leaves the row untouched; it WILL be offered again
#     next time this tool runs.
#   - "Stop" — ends the session (every action already autosaves).

import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image

corr_state = {"pos": 0, "resolved_this_session": 0}

image_out2 = widgets.Output()
status_out2 = widgets.Output()
text_box2 = widgets.Text(layout=widgets.Layout(width="300px"))
btn_submit2 = widgets.Button(description="Submit", button_style="success")
btn_confirm2 = widgets.Button(description="Confirm as-is", button_style="warning")
btn_skipfornow2 = widgets.Button(description="Skip for now")
btn_stop2 = widgets.Button(description="Stop", button_style="danger")

controls2 = widgets.HBox([text_box2, btn_submit2, btn_confirm2, btn_skipfornow2, btn_stop2])
display(image_out2, controls2, status_out2)


def save_all_rows():
    """Overwrites manual_labels.csv in place with the current in-memory rows.
    Rewriting the whole file (not appending) is what makes this an in-place
    update rather than a duplicate-row append. Rewritten after every single
    action so a crash mid-session loses at most nothing already confirmed."""
    with open(OUTPUT_CSV, "w", newline='') as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        writer.writeheader()
        writer.writerows(all_rows)


def show_current_correction():
    with image_out2:
        clear_output(wait=True)
        if corr_state["pos"] >= len(batch_indices):
            print(f"Batch finished. Resolved {corr_state['resolved_this_session']} rows this session.")
            return
        row = all_rows[batch_indices[corr_state["pos"]]]
        img_path = row["image_path"]
        x_c, y_c, w, h = float(row["x_center"]), float(row["y_center"]), float(row["width"]), float(row["height"])
        img = Image.open(img_path).convert("RGB")
        W, H = img.size
        left = (x_c - w / 2) * W
        top = (y_c - h / 2) * H
        right = (x_c + w / 2) * W
        bottom = (y_c + h / 2) * H
        margin = 0.05
        left = max(0, left - margin * (right - left))
        right = min(W, right + margin * (right - left))
        top = max(0, top - margin * (bottom - top))
        bottom = min(H, bottom + margin * (bottom - top))
        crop = img.crop((left, top, right, bottom))
        display(crop)
        print(f"{os.path.basename(img_path)}  (box {row['box_index']})  [{row['split']}]"
              f"  —  {corr_state['pos'] + 1}/{len(batch_indices)}")
        print(f"Old label: {row['label_text']}   (checksum_valid=False)")
    text_box2.value = row["label_text"]


def advance_correction():
    corr_state["pos"] += 1
    show_current_correction()


def on_submit2(_):
    idx = batch_indices[corr_state["pos"]]
    entry = text_box2.value.strip()
    if not entry:
        return
    is_valid, msg = validate_full_code(entry)
    with status_out2:
        clear_output(wait=True)
        if not is_valid:
            print(f"⚠️  {msg} — saved anyway, still checksum_valid=False, will resurface next pass.")
        else:
            print("✓ Fixed (checksum_valid=True)")
    all_rows[idx]["label_text"] = entry.upper()
    all_rows[idx]["checksum_valid"] = str(is_valid)
    all_rows[idx]["timestamp"] = datetime.now().isoformat()
    # status stays "labeled" either way -- corrected rows re-enter the normal
    # pool; only "Confirm as-is" changes status.
    save_all_rows()
    corr_state["resolved_this_session"] += 1
    advance_correction()


def on_confirm2(_):
    idx = batch_indices[corr_state["pos"]]
    with status_out2:
        clear_output(wait=True)
        print("Confirmed as-is — genuinely malformed real-world code, won't be offered again.")
    all_rows[idx]["status"] = "confirmed_asis"
    all_rows[idx]["timestamp"] = datetime.now().isoformat()
    save_all_rows()
    corr_state["resolved_this_session"] += 1
    advance_correction()


def on_skipfornow2(_):
    advance_correction()


def on_stop2(_):
    with status_out2:
        clear_output(wait=True)
        print(f"Stopped. Resolved {corr_state['resolved_this_session']} rows this session.")


btn_submit2.on_click(on_submit2)
btn_confirm2.on_click(on_confirm2)
btn_skipfornow2.on_click(on_skipfornow2)
btn_stop2.on_click(on_stop2)
text_box2.on_submit(on_submit2)

show_current_correction()

Output()

Output()

In [32]:
# ---- CELL 6: Sanity check after a correction session ----
import pandas as pd
df = pd.read_csv(OUTPUT_CSV)
print(f"Total rows in CSV: {len(df)}")
print(df['status'].value_counts())
remaining_false = df[(df['status'] == 'labeled') & (df['checksum_valid'].astype(str) == 'False')]
print(f"Still unresolved (checksum_valid=False, status=labeled): {len(remaining_false)}")

Total rows in CSV: 2012
status
labeled           1979
skip                18
clipped              9
confirmed_asis       6
Name: count, dtype: int64
Still unresolved (checksum_valid=False, status=labeled): 0
